# [Step 5 - PyPDFLoader] Page-by-page PDF ingestion

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

### What you'll learn

- how `PyPDFLoader` turns each PDF PAGE into one Document
- what `metadata["source"]` and `metadata["page"]` give you for free
- a real manipulation: concatenating selected pages into one text block
- why pypdf's local parsing needs no external service - and where it stops

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================

# --- Standard library -------------------------------------------------
import os                      # file-system odds and ends
import urllib.request          # polite HTTP fetching
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import PyPDFLoader

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - fetch once, cache under DATA, reuse forever.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_5144\4255773782.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### 1. Why PDFs are their own beast

A PDF is a binary LAYOUT format: glyphs are positioned on pages, not stored
as flowing text. Loaders must reverse-engineer reading order - which is why
PDF loading is never byte-perfect and always PAGE-based.

`PyPDFLoader` wraps the pure-Python `pypdf` library:

- **granularity**: one Document per PAGE;
- **metadata**: `{"source": <path>, "page": <0-based index>}` - built in,
  exactly what citations need later;
- **zero services**: extraction happens locally in Python. Contrast the
  `UnstructuredPDFLoader` route, which wants system-level dependencies.

Specimen: *Attention Is All You Need* (arXiv 1706.03761) - the Transformer
paper. Fitting: you are two modules away from using its architecture daily.

In [2]:
ATTENTION_PDF_URL = "https://arxiv.org/pdf/1706.03761"
pdf_path = DATA / "attention.pdf"

payload = get_bytes("attention.pdf", ATTENTION_PDF_URL)   # download-once
print("magic bytes:", payload[:5])                        # expect b'%PDF-'
if not payload.startswith(b"%PDF"):
    raise ValueError("Downloaded file does not look like a PDF - aborting.")

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

print(f"\nDocuments (pages) loaded : {len(docs)}")
print(f"first page metadata      : {docs[0].metadata}")
print(f"second page metadata     : {docs[1].metadata}")

print("\n--- page 1 preview (first 300 chars) ---")
print(to_ascii(docs[0].page_content[:300]))

[cache] attention.pdf: 147,012 bytes
magic bytes: b'%PDF-'



Documents (pages) loaded : 9
first page metadata      : {'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'LaTeX with hyperref package', 'creationdate': '2017-11-07T20:29:59-05:00', 'moddate': '2017-11-07T20:29:59-05:00', 'title': 'Emergent Anyon Distribution in the Unruh Effect', 'subject': '', 'author': 'Satoshi Ohya', 'keywords': 'Unruh effect; anyon; conformal field theory', 'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\attention.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}
second page metadata     : {'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'LaTeX with hyperref package', 'creationdate': '2017-11-07T20:29:59-05:00', 'moddate': '2017-11-07T20:29:59-05:00', 'title': 'Emergent Anyon Distribution in the Unruh Effect', 'subject': '', 'author': 'Satoshi Ohya', 'keywords': 'Unruh effect; anyon; conformal field theory', 'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\attention.pdf', 'total_pages':

### 3. Inspect: what did pypdf actually recover?

Two things worth measuring on every PDF corpus:

In [3]:
total_chars = sum(len(doc.page_content) for doc in docs)
empty_pages = [doc.metadata["page"] for doc in docs if not doc.page_content.strip()]
avg_chars = total_chars / max(len(docs), 1)

print(f"pages            : {len(docs)}")
print(f"total characters : {total_chars:,}  (average {avg_chars:,.0f} per page)")
print(f"empty pages      : {empty_pages if empty_pages else 'none'}")

# Character-count spread across pages tells us whether extraction worked
# uniformly - wildly uneven numbers hint at figures-only or scanned pages.
sizes = [len(doc.page_content) for doc in docs]
print(f"chars per page   : min={min(sizes)}, max={max(sizes)}")

pages            : 9
total characters : 25,971  (average 2,886 per page)
empty pages      : none
chars per page   : min=124, max=4272


### 4. One meaningful manipulation: stitch selected pages

The abstract + introduction span pages 1-2 (indexes 0-1) but sentences run
across the page break. For questions aimed at ONE region it pays to fuse
those pages into a single text block - while keeping the page range in
metadata for citation.

In [4]:
selected = docs[0:3]                                   # pages 1..3 (0-based)
combined_text = "\n\n".join(page.page_content for page in selected)
combined_doc = type(docs[0])(
    page_content=combined_text,
    metadata={
        "source": selected[0].metadata["source"],
        "pages": [page.metadata["page"] for page in selected],  # provenance!
    },
)

print(f"fused {len(selected)} pages into one Document "
      f"({len(combined_text):,} chars)")
print(f"metadata: {combined_doc.metadata}")
print("\n--- seam check: end of page 1 vs start of page 2 ---")
print(to_ascii(selected[0].page_content[-90:]))
print("  [...page boundary...]")
print(to_ascii(selected[1].page_content[:90]))

fused 3 pages into one Document (10,786 chars)
metadata: {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\attention.pdf', 'pages': [0, 1, 2]}

--- seam check: end of page 1 vs start of page 2 ---
th non-vanishing anomalous dimension, the detector? s power spectrum generally obeys the
1
  [...page boundary...]
thermal distribution for (1 + 1)-dimensional anyons that is derived by Liguori, Mintchev ,


### 5. Pitfalls worth remembering

**Pitfall - page != section**: ideas flow across page boundaries, so
page-sized Documents slice mid-thought. That is acceptable at INGEST time -
module 06 re-chunks with overlap - but never ship raw pages as final RAG units.

**Pitfall - scanned PDFs return empty strings**: `pypdf` reads embedded TEXT
only; image-only pages need OCR FIRST (e.g. tesseract) before any loader.
Our `empty pages` check above is your early-warning system.

**Pro-tip**: multi-column layouts can scramble reading order under pypdf.
When extraction looks jumbled, compare alternatives (`pdfplumber`, cloud
extractors) before blaming your pipeline logic.

### Takeaway

**PyPDFLoader = one Document per page with source+page metadata, extracted
locally by pypdf. Inspect char counts for silent failures, then fuse or
split pages deliberately - page metadata stays your citation anchor.**

### Summary

- The Transformer paper became N page-Documents (N printed above), each
  carrying `{"source", "page"}` out of the box.
- Sanity metrics (totals, averages, empty-page list, min/max) expose broken
  extractions before they reach embeddings.
- Concatenating selected pages - with the page list recorded in metadata -
  shows manipulation WITHOUT losing provenance.
- Local extraction, zero services; know its limits (scans, columns, layout).